# Brain Tumor MRI Classification using PyTorch

---
#### DISCLAIMER: THIS IS NOT MEDICAL ADVICE, DO NOT SELF DIAGNOSE

## Imports and loading data

In [1]:
import torch
from torch import nn, optim
from torch.nn import Flatten
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

# use gpu if it's available if not cpu which is much slower
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Image transformations
Certain transformations are needed before using an image, e.g resize, changing to tensor and normalizing

In [2]:
# Every image will go through this transformation pipeline
pic_trans = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [3]:
train_dl = DataLoader(
    datasets.ImageFolder('data/Training', pic_trans),
    batch_size=32,
    shuffle=True,
    num_workers=4, # speeds up loading data
    pin_memory=True
)

test_dl = DataLoader(
    datasets.ImageFolder('data/Testing', pic_trans),
    batch_size=32,
    shuffle=False, # there is no need to shuffle, as this is just for evaluation
    num_workers=4, # speeds up loading data
    pin_memory=True
)

## Define model

In [4]:
model = nn.Sequential(
    nn.Conv2d(3, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(128 * 16 * 16, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 4) # final layer, which takes 256 as input and 4 as output (4 classes
).to(device)

In [5]:
opt = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

## Training loop

In [ ]:
for epoch in range(25):
    running_loss = 0

    for x, y in train_dl:
        opt.zero_grad()

        loss = loss_fn(model(x.to(device)), y.to(device))
        loss.backward()

        running_loss += loss.item()
        opt.step()

    print(f'Epoch {epoch+1}: Loss was {running_loss}')

Epoch 1: Loss was 129.99363136291504


In [ ]:
model.eval()
test_loss, correct = 0.0, 0
with torch.no_grad():
    for x, y in test_dl:
        y, y = x.to(device), y.to(device)

        logits = model(x)
        test_loss = loss_fn(logits, y).item() * y.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == y).float().sum().item()

test_loss /= len(test_dl.dataset)
accuracy = 100.0 * correct / len(test_dl.dataset)

print(f'Test set: Average loss: {test_loss}, Accuracy: {accuracy}')

Decent with room for improvement ;))

## Visualising image

In [ ]:
import random
from torchvision.transforms.functional import to_pil_image

model.eval()

idx = random.randrange(len(test_dl.dataset))
img, label = test_dl.dataset[idx]

unnorm = img * 0.5 + 0.5
plt.imshow(to_pil_image(unnorm))
plt.axis('off')
plt.title('Sample picture')
plt.show()

with torch.no_grad():
    logits = model(img.unsqueeze(0).to(device))
    pred = logits.argmax(dim=1).item()

class_names = test_dl.dataset.classes
print(f'Predicted: {class_names[pred]}')
print(f'Actual: {class_names[label]}')

## Conclusion
Model is decent but there is room for improvement. Hyperparameter tuning, trying different architectures and optimisers, learning rates, model checkpointing.

## Model tuning

In [ ]:
from torch.utils.data import random_split

IMG = 256
EPOCHS = 8

pic_trans = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 85/15 train/validation split, test set kept for the very end
full = datasets.ImageFolder('data/Training', pic_trans)
n_val = int(0.15 * len(full))
train_ds, val_ds = random_split(full, [len(full) - n_val, n_val], generator=torch.Generator().manual_seed(42))
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
val_dl = DataLoader(val_ds, batch_size=32, num_workers=4)
test_dl = DataLoader(datasets.ImageFolder('data/Testing', pic_trans), batch_size=32, num_workers=4)

In [ ]:
def make_model():
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(128 * (IMG // 8) ** 2, 256), nn.ReLU(), nn.Dropout(0.5),  # 3 poolings: 256 -> 32
        nn.Linear(256, 4)
    ).to(device)

In [ ]:
def accuracy(model, dl):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    return correct / len(dl.dataset)

optimisers = {
    'Adam': lambda p, lr: optim.Adam(p, lr=lr),
    'SGD': lambda p, lr: optim.SGD(p, lr=lr, momentum=0.9),
    'RMSprop': lambda p, lr: optim.RMSprop(p, lr=lr),
}

learning_rates = [1e-2, 1e-3, 1e-4]
loss_fn = nn.CrossEntropyLoss()
results, best_acc = [], 0

In [ ]:
for opt_name, make_opt in optimisers.items():
    for lr in learning_rates:
        torch.manual_seed(42)
        model = make_model()
        opt = make_opt(model.parameters(), lr)
        run_best = 0

        for epoch in range(EPOCHS):
            model.train()
            for x, y in train_dl:
                opt.zero_grad()
                loss = loss_fn(model(x.to(device)), y.to(device))
                loss.backward()
                opt.step()

            val_acc = accuracy(model, val_dl)
            run_best = max(run_best, val_acc)
            if val_acc > best_acc:  # checkpoint: best model across all runs
                best_acc = val_acc
                best_setup = (opt_name, lr, epoch + 1)
                torch.save(model.state_dict(), 'best_model.pt')

        print(f'{opt_name}, lr={lr}: best val acc {run_best:.4f}')
        results.append({'optimiser': opt_name, 'lr': lr, 'val_acc': run_best})

print(pd.DataFrame(results).pivot(index='optimiser', columns='lr', values='val_acc'))

In [ ]:
model = make_model()
model.load_state_dict(torch.load('best_model.pt'))
print(f'Best: {best_setup[0]}, lr={best_setup[1]}, epoch {best_setup[2]} | test acc {accuracy(model, test_dl):.4f}')